# DyFHL-PA — full experiments (Colab)
Reproduces the evaluation of *DyFHL-PA: Governing Change in Cross-Organisation Agent and Tool Ecosystems with Federated Dynamic Policy Hypergraphs* (Section 8, RQ1–RQ7).

| RQ | Question | Script | Runtime (CPU) |
|---|---|---|---|
| RQ1 | Incremental admission = full recomputation; secure = plaintext | `federation_eval.py`, tests | ~5 min |
| RQ2 | One year of AWS managed-policy changes | `extract_history.py`, `aws_history.py` | ~10 min |
| RQ3 | What MCP tools let owners declare | `census_tools.py` | ~3 min |
| RQ4–RQ6 | Scaling, secure cost, probing leakage | `federation_eval.py` | ~10 min |
| RQ5 (static) | Exact secure checks vs sketch on earlier benchmarks | `secure_exact.py` | ~5 min |
| RQ7 | Learned ranking (Amazon access) | `kill_test_demand.py` | ~15 min |

Then figures, tables and number macros are regenerated and the paper is recompiled.

In [ ]:
!git clone -q https://github.com/msellamiTN/DyFHL-PC.git
%cd DyFHL-PC
!pip -q install numpy scipy scikit-learn torch matplotlib pandas pytest

## 0. Property tests (lemmas, duality, attenuation, incremental maintenance, coNP reduction, admission, secure checks)

In [ ]:
!python3 tests/test_composed.py
!python3 tests/test_v4_model.py
!python3 tests/test_federation.py

## RQ2 — AWS managed-policy history
Use the shipped extract (`data/history/*.gz`), or set `REFETCH=True` to rebuild it from the MAMIP git history (3 Nov 2025 – 14 Nov 2026).

In [ ]:
REFETCH = False
import os
if REFETCH:
    !git clone -q --shallow-since=2025-11-01 https://github.com/z0ph/MAMIP.git /content/mamip
    !python3 -I scripts/extract_history.py /content/mamip data/history
else:
    !gunzip -kf data/history/events.jsonl.gz data/history/base.json.gz
!python3 -I scripts/aws_history.py data/history results/aws_history.json

## RQ3 — MCP Atlas Census (229,827 tool schemas, CC0, ~90 MB)

In [ ]:
!wget -q https://github.com/DSHCorrectover/mcp-atlas-census/releases/download/v1.0.0/server_tools.jsonl.gz -O /content/tools.gz
!python3 -I scripts/census_tools.py /content/tools.gz results/census_tools.json

## RQ1, RQ4, RQ5, RQ6 — constructed federations
10 seeds × 300 events (exactness), scaling grid K∈{2,4,8,16} and 50–3,200 rules per organisation (3 seeds × 200 events), secure cost per event vs full re-check, probing leakage.

In [ ]:
!python3 scripts/federation_eval.py results/federation.json

## RQ5 (static) — exact secret-shared checks on the earlier benchmarks (5 seeds)
The sketch-baseline numbers are read from `results/revision_tops.json`.

In [ ]:
!python3 scripts/secure_exact.py 0,1,2,3,4

## RQ7 — expected-use ranking on real access requests (8 organisations, 5 seeds)

In [ ]:
!python3 scripts/kill_test_demand.py

## Figures, tables, numbers and paper

In [ ]:
!python3 scripts/make_figures_v2.py
!python3 scripts/make_concept_figures.py
!apt-get -qq install -y texlive-latex-extra texlive-science > /dev/null
%cd paper
!pdflatex -interaction=nonstopmode main.tex > /dev/null; bibtex main > /dev/null; pdflatex -interaction=nonstopmode main.tex > /dev/null; pdflatex -interaction=nonstopmode main.tex | grep -E "^!|undefined" ; ls -l main.pdf
%cd ..

## Summary of key numbers

In [ ]:
import json, numpy as np
a = json.load(open('results/aws_history.json'))['events']
fr = np.array([r['region_statements']/r['total_statements'] for r in a])
print('AWS events', len(a), '| region median %.1f%%' % (100*np.median(fr)))
f = json.load(open('results/federation.json'))
print('RQ1 mismatches', sum(x['mismatches'] for x in f['e1']), '| secure checks agree',
      sum(x['secure_pairwise_agree']+x['secure_cross_agree'] for x in f['e1']), '/', sum(x['secure_pairwise']+x['secure_cross'] for x in f['e1']))
for r in f['e5']: print('K=%d rules=%d full %.1f MB, event mean %.0f kB' % (r['K'], r['rules'], r['full_MB'], 1e3*r['event_MB_mean']))